# nb_90_run_pipeline

The **driver**. It runs the whole data product in order and records what happened to every step.

Steps, in order: `nb_10` bronze, `nb_20` silver clean, `nb_21` silver sales, `nb_21b` history and retention, `nb_22` identity, `nb_30` gold.
After this notebook, the Fabric pipeline loads the gold tables into the Warehouse.

What it does
1. Runs each notebook one after the other, each with its own time limit.
2. **Stops at the first failure.** Later steps are recorded as `skipped`, not silently missing.
3. Sorts the failure into a **class** with a suggested action (see the table below), so the person on call knows where to look first.
4. Writes one row per step to `ctl_pipeline_run`. The Pipeline Health report reads that table.
5. Fails itself with a clear message, so the pipeline can send an alert.

| Failure class | Meaning | Usually |
|---|---|---|
| `source` | the input is wrong or missing | wait for or ask for a corrected file; do not retry blindly |
| `integration` | two parts do not fit (missing table, changed column, schema) | fix the dependency or the mapping |
| `processing` | our own check failed (reconciliation, integrity) | a bug or a rule problem; read the check output |
| `infrastructure` | the platform failed (timeout, memory, capacity) | retry is usually safe |
| `unknown` | none of the above matched | triage by hand |

The parameters in the first code cell are set by the pipeline. Running the notebook by hand uses the defaults.


In [ ]:
# ---- Parameters (the pipeline overrides these) ----------------------------------------
PIPELINE_NAME = "maison-memory-daily"
PIPELINE_RUN_ID = ""            # the pipeline passes @pipeline().RunId. Empty = made up here.

# For a failure drill only: "step_name|error text". The named step is NOT run; it fails with that text instead.
# The row is marked is_simulated = 1 so a drill can never be mistaken for a real failure.
SIMULATE_FAILURE = ""

In [ ]:
# ---- The steps and their time limits (seconds) ----------------------------------------
STEPS = [
    ("nb_10_bronze_ingest", 3600),
    ("nb_20_silver_clean", 3600),
    ("nb_21_silver_sales", 3600),
    ("nb_21b_silver_history_retention", 3600),
    ("nb_22_silver_identity", 3600),
    ("nb_30_gold_build", 3600),
]

In [ ]:
# ---- Failure classification -----------------------------------------------------------
# Order matters: our own check messages come first, then the platform, then bad input, then broken fit.
CLASS_RULES = [
    ("processing", ["reconciliation failed", "checks failed", "more than one row", "difference ", "reconcile"]),
    ("infrastructure", ["timed out", "timeout", "outofmemory", "out of memory", "capacity", "throttl", "too many requests",
                        "status code 429", "status code 503", "http 429", "http 503", "service unavailable", "connection reset",
                        "livy", "session expired", "executor lost", "tokenservice", "fetch notebook content"]),
    ("source", ["filenotfound", "file not found", "no such file", "path does not exist", "does not exist: ", "short_file",
                "missing_file", "corrupt", "malformed", "unable to infer schema", "cannot parse", "is not a valid csv"]),
    ("integration", ["table_or_view_not_found", "table or view not found", "cannot be found", "unresolved_column", "cannot resolve",
                     "ambiguous_reference", "mismatch", "incompatible", "schema", "datatype", "cannot cast", "no such table"]),
]
SUGGESTED_ACTION = {
    "source": "Check the landing files for the batch (ctl_file_log). Ask the source team for a corrected or complete file. Do not retry until the input changes.",
    "integration": "A table, column or type does not match what this step expects. Check that the earlier steps finished and that no schema changed.",
    "processing": "One of our own checks failed. Read the check output of this notebook (reconciliation or integrity cell) before changing anything.",
    "infrastructure": "The platform failed, not the data. Retry once. If it repeats, check capacity and the Spark session.",
    "unknown": "No rule matched. Read the full error text and classify it by hand, then add a rule here.",
}
RETRYABLE = {"infrastructure": 1, "source": 0, "integration": 0, "processing": 0, "unknown": 0}


def classify_failure(error_text):
    """Return (failure_class, retryable 0/1, suggested_action) for an error message."""
    t = (error_text or "").lower()
    for cls, words in CLASS_RULES:
        if any(w in t for w in words):
            return cls, RETRYABLE[cls], SUGGESTED_ACTION[cls]
    return "unknown", RETRYABLE["unknown"], SUGGESTED_ACTION["unknown"]


print("failure classes:", [c for c, _ in CLASS_RULES] + ["unknown"])

In [ ]:
# ---- Setup ----------------------------------------------------------------------------
import uuid
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType, TimestampType)

if not PIPELINE_RUN_ID:
    PIPELINE_RUN_ID = "manual-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]
print("pipeline run id:", PIPELINE_RUN_ID)


def now_utc():
    return datetime.now(timezone.utc).replace(tzinfo=None)


RUN_SCHEMA = StructType([
    StructField("pipeline_run_id", StringType()), StructField("pipeline_name", StringType()),
    StructField("step_order", IntegerType()), StructField("step", StringType()), StructField("status", StringType()),
    StructField("started_at", TimestampType()), StructField("finished_at", TimestampType()), StructField("duration_s", LongType()),
    StructField("failure_class", StringType()), StructField("retryable", IntegerType()),
    StructField("error_message", StringType()), StructField("suggested_action", StringType()),
    StructField("is_simulated", IntegerType())])
rows = []


def record(order, step, status, started, finished, cls=None, retryable=None, error=None, action=None, simulated=0):
    secs = int((finished - started).total_seconds()) if started and finished else 0
    rows.append(Row(pipeline_run_id=PIPELINE_RUN_ID, pipeline_name=PIPELINE_NAME, step_order=order, step=step, status=status,
                    started_at=started, finished_at=finished, duration_s=secs, failure_class=cls, retryable=retryable,
                    error_message=(error or "")[:1500] or None, suggested_action=action, is_simulated=simulated))


def save_rows():
    spark.createDataFrame(rows, schema=RUN_SCHEMA).write.format("delta").mode("append") \
         .option("mergeSchema", "true").saveAsTable("ctl_pipeline_run")

In [ ]:
# ---- Run the steps --------------------------------------------------------------------
# A failure is caught, classified and recorded. The remaining steps are recorded as skipped.
# The error is raised again at the end, so the pipeline sees a failed activity and can alert.
sim_step, _, sim_text = SIMULATE_FAILURE.partition("|")
failure = None
run_started = now_utc()

for order, (name, timeout) in enumerate(STEPS, start=1):
    if failure is not None:
        t = now_utc()
        record(order, name, "skipped", t, t)
        continue
    started = now_utc()
    try:
        if sim_step == name:
            raise RuntimeError(sim_text or "simulated failure")
        notebookutils.notebook.run(name, timeout)
        record(order, name, "success", started, now_utc(), simulated=0)
        print(f"{order}. {name}: success")
    except Exception as e:
        cls, retry, action = classify_failure(str(e))
        record(order, name, "failed", started, now_utc(), cls, retry, str(e), action, simulated=1 if sim_step == name else 0)
        failure = (name, cls, str(e))
        print(f"{order}. {name}: FAILED [{cls}]")

overall = "failed" if failure else "success"
record(0, "_pipeline", overall, run_started, now_utc(),
       failure[1] if failure else None, RETRYABLE.get(failure[1]) if failure else None,
       failure[2] if failure else None, SUGGESTED_ACTION.get(failure[1]) if failure else None,
       simulated=1 if (failure and sim_step == failure[0]) else 0)
save_rows()
print("pipeline", overall, "| rows written to ctl_pipeline_run:", len(rows))
if failure:
    raise RuntimeError(f"[{failure[1]}] step {failure[0]} failed: {failure[2][:600]}")

## Check the result
Read-only cells.

In [ ]:
# This run, step by step
(spark.table("ctl_pipeline_run").filter(f"pipeline_run_id = '{PIPELINE_RUN_ID}'")
      .select("step_order", "step", "status", "duration_s", "failure_class", "retryable", "is_simulated")
      .orderBy("step_order").show(truncate=False))

In [ ]:
# The last 10 pipeline runs
(spark.table("ctl_pipeline_run").filter("step = '_pipeline'").orderBy("started_at", ascending=False)
      .select("pipeline_run_id", "status", "started_at", "duration_s", "failure_class", "is_simulated").show(10, truncate=False))